# 43. Factorized densities and Gaussian constraints

**Objectives:**
- Multiply a Dalitz-plot density by an independent 1D discriminant PDF via `FactorizedDensity`.
- Turn a complex dynamics lineshape (`RelativisticBreitWigner`) into a normalized 1D intensity
  PDF via `LineshapeIntensity1D`.
- Add a `GaussianConstraint` on a fit parameter to an NLL via `ConstrainedNLL`, and check the
  constraint changes the objective by exactly `0.5*((x-mu)/sigma)^2`.

Run cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero. See `docs/discriminants_and_constraints.md`.


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes and PDFs use complex128/float64.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    ConstrainedNLL, DecayChannel, DecayModel, FactorizedDensity, Gaussian1D,
    GaussianConstraint, LineshapeIntensity1D, NonResonant, Parameter, RealImag,
    RelativisticBreitWigner, Resonance, ResonanceContext, generate_toy,
)

## 1. A small Dalitz model and toy sample

The same rho(770) + non-resonant `D+ -> pi- pi+ pi+` toy model used elsewhere in this course.
`NR.x`/`NR.y` are floatable Cartesian coefficients (`Parameter.coefficient`, not a bare float --
bare floats cannot be floated in a fit).

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(model, 600, parameters=truth, seed=7, inverse_resolution=256)
print(f"Generated {data.size} events")

Generated 600 events


## 2. `FactorizedDensity`: Dalitz density x discriminant PDF

`model.pdf()` is a `SignalPDF`: a normalized Dalitz density that consumes `data.as_dict()`
(`{"s12", "s13", "s23"}`) and a parameter mapping. `FactorizedDensity` multiplies it by an
independent 1D PDF over a synthetic discriminating variable (here a reconstructed parent mass,
`Gaussian1D` on a narrow window) evaluated on the *same* events. Since the discriminant PDF
integrates to 1 over its own declared window (`docs/discriminants_and_constraints.md`), the
factorized density at fixed Dalitz coordinates integrates to the same value the base density
already had there -- the two factors normalize independently.

In [3]:
base_density = model.pdf()
data_dict = data.as_dict()

rng = np.random.default_rng(11)
reco_mass = rng.normal(loc=5.279, scale=0.015, size=data.size)

mass_pdf = Gaussian1D(mean=5.279, sigma=0.015, low=5.20, high=5.35)
factorized = FactorizedDensity(
    base_density=lambda parameters: base_density(data_dict, parameters),
    observables={"mass": jnp.asarray(reco_mass)},
    pdfs={"mass": mass_pdf},
)

base_values = np.asarray(base_density(data_dict, truth))
factorized_values = np.asarray(factorized(truth))
mass_factor = np.asarray(mass_pdf(reco_mass, truth))

print("base density [:5]      =", base_values[:5])
print("mass pdf factor [:5]   =", mass_factor[:5])
print("factorized density[:5] =", factorized_values[:5])
assert np.allclose(factorized_values, base_values * mass_factor)

x_mass = np.linspace(5.20, 5.35, 2000)
mass_integral = np.trapezoid(np.asarray(mass_pdf(x_mass, truth)), x_mass)
print(f"mass PDF integral over its window = {mass_integral:.6f}")
assert abs(mass_integral - 1.0) < 1e-4

base density [:5]      = [0.47982085 0.15290484 0.09921244 0.17303702 0.69119812]
mass pdf factor [:5]   = [26.58064039 10.55195374 12.56351349 23.34916124 25.44132457]
factorized density[:5] = [12.75394548  1.61344479  1.24645684  4.04026919 17.58499561]


mass PDF integral over its window = 1.000000


## 3. `LineshapeIntensity1D`: a dynamics lineshape as a 1D intensity PDF

`RelativisticBreitWigner` is the same lineshape used inside `Resonance`, ported to a normalized
1D PDF `|R(m)|^2 / integral |R(m')|^2 dm'` over the full physical two-body mass interval
(`docs/convolution_resolution.md`).

In [4]:
context = ResonanceContext(
    parent_mass=channel.parent_mass,
    daughter_masses=(channel.daughter_masses[0], channel.daughter_masses[1]),
    bachelor_mass=channel.daughter_masses[2],
    spin=1,
    pole_mass=0.7753,
    pole_width=0.1491,
    resonance_radius=4.0,
    parent_radius=4.0,
)
rho_mass_pdf = LineshapeIntensity1D.from_context(
    RelativisticBreitWigner(), context, quadrature_order=512,
)

x_rho = np.linspace(rho_mass_pdf.low, rho_mass_pdf.high, 4000)
y_rho = np.asarray(rho_mass_pdf(x_rho))
rho_integral = np.trapezoid(y_rho, x_rho)
print(f"low={rho_mass_pdf.low:.4f} GeV, high={rho_mass_pdf.high:.4f} GeV")
print(f"LineshapeIntensity1D integral over [low, high] = {rho_integral:.6f}")
assert abs(rho_integral - 1.0) < 1e-3

low=0.2791 GeV, high=1.7301 GeV
LineshapeIntensity1D integral over [low, high] = 1.000000


## 4. `GaussianConstraint` + `ConstrainedNLL`

Build a plain NLL from `SignalPDF.logpdf` on the toy data, then attach an external Gaussian
constraint on `NR.x`. `GaussianConstraint.parameter` must be an object with `.resolve(parameters)`
-- a fit `Parameter` itself (here `x`), not a bare string name. The constraint adds
`0.5*((value-mean)/sigma)^2` up to an additive constant; the difference between the constrained
and unconstrained objective at *any* parameter point must equal exactly that penalty, evaluated
at that point's `NR.x` value.

In [5]:
def base_nll(parameters):
    return -jnp.sum(base_density.logpdf(data_dict, parameters))

constraint = GaussianConstraint(x, mean=0.70, sigma=0.05)
constrained_nll = ConstrainedNLL(base_nll, constraint)

displaced = dict(truth)
displaced["NR.x"] = 0.40  # away from both the constraint mean and the toy truth

unconstrained_value = float(base_nll(displaced))
constrained_value = float(constrained_nll(displaced))
penalty = float(constraint(displaced))
expected_penalty = 0.5 * ((displaced["NR.x"] - 0.70) / 0.05) ** 2

print(f"NLL(displaced)            = {unconstrained_value:.6f}")
print(f"ConstrainedNLL(displaced) = {constrained_value:.6f}")
print(f"constraint penalty        = {penalty:.6f}")
print(f"0.5*((x-mu)/sigma)^2      = {expected_penalty:.6f}")
assert abs(penalty - expected_penalty) < 1e-10
assert abs((constrained_value - unconstrained_value) - expected_penalty) < 1e-10

NLL(displaced)            = 730.858211
ConstrainedNLL(displaced) = 748.858211
constraint penalty        = 18.000000
0.5*((x-mu)/sigma)^2      = 18.000000


## Summary and exercises

1. Move `constraint.mean` to the toy truth value of `NR.x` and see the penalty at the truth
   point drop to (nearly) zero, while a displaced point still pays the full quadratic penalty.
2. Add a second `GaussianConstraint` (e.g. on `NR.y`) to the same `ConstrainedNLL` call --
   `ConstrainedNLL` accepts any number of constraints and sums their penalties.
3. Replace the synthetic `reco_mass` array with a real reconstructed-mass branch when reading
   data from ROOT (`docs/root_io.md`) to build a genuine signal-plus-sideband discriminant model.

Reference: `docs/discriminants_and_constraints.md`.

Return to [the course guide](TUTORIALS.md).
